# Day 1 — Document Ingestion & Indexing
### AI Clinical Decision Support Lite · AI Max Team

This notebook demonstrates our **Day 1 pipeline** on two premier WHO hypertension guidelines:
- `Guideline for the pharmacological treatment of hypertension in adults.pdf` (61 pages)
- `WHO-NMH-NVI-18.2-eng.pdf` (43 pages)

**Pipeline:** PDF parse → header cleaning → section-aware chunking → `bge-small` embeddings → ChromaDB.

All functions import directly from `ingest.py` and `config.py` in the repo root.

## 0. Setup

In [1]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "notebooks").exists() and not (REPO_ROOT / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent if REPO_ROOT.name == "day1" else REPO_ROOT
if not (REPO_ROOT / "config.py").exists():
    REPO_ROOT = Path(os.path.abspath("../../"))
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import config
from pathlib import Path

print("✅ Environment ready — AI Max Team pipeline")
print(f"   Repo root         : {REPO_ROOT}")
print(f"   Data directory    : {config.DATA_DIR}")
print(f"   Vector store      : {config.VECTOR_DB_DIR}")
print(f"   Chunk size/overlap: {config.CHUNK_SIZE}/{config.CHUNK_OVERLAP} tokens (~15% overlap)")
print(f"   Embedding model   : {config.EMBEDDING_MODEL_NAME}")
print(f"   TOP_K (Day 2)     : {config.TOP_K}")
print()
pdfs = sorted(config.DATA_DIR.glob("*.pdf"))
print(f"   Active PDFs ({len(pdfs)}):")
for p in pdfs:
    print(f"     • {p.name} ({p.stat().st_size / 1_048_576:.1f} MB)")
ref = sorted(config.REFERENCE_DIR.glob("*.pdf"))
print(f"   Reference annexes in reference_candidates/: {len(ref)} file(s) (not indexed)")

✅ Environment ready — AI Max Team pipeline
   Repo root         : C:\Users\Maka\Downloads\ai-clinical-decision-support-day1-day-2-main\ai-clinical-decision-support-day1-day-2-main
   Data directory    : C:\Users\Maka\Downloads\ai-clinical-decision-support-day1-day-2-main\ai-clinical-decision-support-day1-day-2-main\data
   Vector store      : C:\Users\Maka\Downloads\ai-clinical-decision-support-day1-day-2-main\ai-clinical-decision-support-day1-day-2-main\vectorstore
   Chunk size/overlap: 500/75 tokens (~15% overlap)
   Embedding model   : BAAI/bge-small-en-v1.5
   TOP_K (Day 2)     : 3

   Active PDFs (2):
     • Guideline for the pharmacological treatment of hypertension in adults.pdf (0.6 MB)
     • WHO-NMH-NVI-18.2-eng.pdf (1.7 MB)
   Reference annexes in reference_candidates/: 5 file(s) (not indexed)


## 1. Why Grounding Matters (Clinical RAG)

| Without RAG | With RAG (our approach) |
|---|---|
| Model may invent recommendations | Model may only cite retrieved WHO text |
| No verifiable source | Every answer traceable to page + chunk_id |
| Stale training data | Official WHO guidelines in `data/` |

## 2. Load & Clean PDF Pages

`load_pdfs()` uses PyPDFLoader and `clean_page_text()` to strip repeated WHO headers/footers before chunking.

In [2]:
from ingest import load_pdfs, clean_page_text

pages = load_pdfs(config.DATA_DIR)
by_doc = {}
for p in pages:
    by_doc[p.metadata["document_name"]] = by_doc.get(p.metadata["document_name"], 0) + 1

print(f"✅ Loaded {len(pages)} pages from {len(by_doc)} WHO guidelines:\n")
for name, count in sorted(by_doc.items()):
    print(f"   • {name}: {count} pages")

sample = pages[10]
print("\n── Sample page metadata ──")
print(f"  document_name : {sample.metadata['document_name']}")
print(f"  page_number   : {sample.metadata['page_number']} (1-indexed)")
print(f"\n── Content preview (after cleaning) ──")
print(sample.page_content[:350])

✅ Loaded 104 pages from 2 WHO guidelines:

   • Guideline for the pharmacological treatment of hypertension in adults.pdf: 61 pages
   • WHO-NMH-NVI-18.2-eng.pdf: 43 pages

── Sample page metadata ──
  document_name : Guideline for the pharmacological treatment of hypertension in adults.pdf
  page_number   : 11 (1-indexed)

── Content preview (after cleaning) ──
WHO recommends a target systolic blood pressure treatment goal of <130 mmHg in patients 
with hypertension and known cardiovascular disease (CVD).
Strong recommendation, moderate-certainty evidence
WHO suggests a target systolic blood pressure treatment goal of <130 mmHg in high-risk 
patients with hypertension (those with high CVD risk, diabetes m


## 3. Section-Aware Chunking (500 / 75 tokens)

`RecursiveCharacterTextSplitter` with paragraph separators. Each chunk gets a **stable per-page** `chunk_id` (`doc_p28_c1`, `doc_p28_c2`, …).

In [3]:
from ingest import chunk_documents

chunks = chunk_documents(pages)
avg_len = sum(len(c.page_content) for c in chunks) // len(chunks)
print(f"✅ {len(chunks)} chunks from {len(pages)} pages (avg ~{avg_len} chars)\n")

from collections import Counter
dist = Counter(c.metadata["document_name"] for c in chunks)
for name, n in dist.items():
    print(f"   • {name[:55]}: {n} chunks")

sample_chunk = chunks[20]
print("\n── Sample chunk metadata ──")
for k, v in sample_chunk.metadata.items():
    print(f"  {k}: {v}")
print("\n── Content preview ──")
print(sample_chunk.page_content[:280])

✅ 190 chunks from 104 pages (avg ~1351 chars)

   • Guideline for the pharmacological treatment of hyperten: 114 chunks
   • WHO-NMH-NVI-18.2-eng.pdf: 76 chunks

── Sample chunk metadata ──
  producer: Adobe PDF Library 10.0.1
  creator: Adobe InDesign CS6 (Macintosh)
  creationdate: 2021-08-24T09:45:25+01:00
  moddate: 2021-08-24T15:53:43+02:00
  trapped: /False
  source: C:\Users\Maka\Downloads\ai-clinical-decision-support-day1-day-2-main\ai-clinical-decision-support-day1-day-2-main\data\Guideline for the pharmacological treatment of hypertension in adults.pdf
  total_pages: 61
  page: 14
  page_label: 3
  document_name: Guideline for the pharmacological treatment of hypertension in adults.pdf
  page_number: 15
  chunk_id: Guideline for the pharmacological treatment of hypertension in adults.pdf_p15_c2

── Content preview ──
refined and voted on during the meeting.
Fig. 1 Analytic framework for antihypertensive medication treatment
Q1: At what BP level should pharmacological 
therapy

## 4. Embed & Persist to ChromaDB

In [4]:
from ingest import build_index

vectordb = build_index(chunks)
count = vectordb._collection.count()
print(f"\n✅ Index persisted: {count} chunks")
print(f"   Collection : {config.COLLECTION_NAME}")
print(f"   Directory  : {config.VECTOR_DB_DIR}")


✅ Index persisted: 190 chunks
   Collection : clinical_guidelines
   Directory  : C:\Users\Maka\Downloads\ai-clinical-decision-support-day1-day-2-main\ai-clinical-decision-support-day1-day-2-main\vectorstore


## 5. Live Clinical Query (Checkpoint 4)

In [5]:
from query import retrieve

query = "What is the target blood pressure for a patient with cardiovascular disease?"
results = retrieve(vectordb, query)

print(f"Query: {query}\n")
for rank, (doc, score) in enumerate(results, 1):
    name = doc.metadata.get("document_name", "?")[:42]
    page = doc.metadata.get("page_number", "?")
    cid = doc.metadata.get("chunk_id", "?")
    conf = "CONFIDENT" if score >= config.CONFIDENCE_THRESHOLD else "UNCERTAIN"
    snippet = doc.page_content.replace("\n", " ")[:70]
    print(f"[{rank}] {conf} score={score:.3f} | p.{page} | {name}")
    print(f"     chunk_id: {cid}")
    print(f"     \"{snippet}...\"\n")

Query: What is the target blood pressure for a patient with cardiovascular disease?

[1] CONFIDENT score=0.793 | p.28 | Guideline for the pharmacological treatmen
     chunk_id: Guideline for the pharmacological treatment of hypertension in adults.pdf_p28_c1
     "3.6 Target blood pressure 6. RECOMMENDATION ON TARGET BLOOD PRESSURES ..."

[2] CONFIDENT score=0.746 | p.14 | WHO-NMH-NVI-18.2-eng.pdf
     chunk_id: WHO-NMH-NVI-18.2-eng.pdf_p14_c1
     "Treatment targets For most patients, blood pressure is considered cont..."

[3] CONFIDENT score=0.740 | p.11 | Guideline for the pharmacological treatmen
     chunk_id: Guideline for the pharmacological treatment of hypertension in adults.pdf_p11_c1
     "WHO recommends a target systolic blood pressure treatment goal of <130..."



## 6. Day 1 Definition of Done

- [x] Two WHO premier guidelines in `data/` (104 pages)
- [x] PDF header cleaning + citation metadata on every chunk
- [x] Stable per-page `chunk_id`
- [x] Local `bge-small-en-v1.5` embeddings (no API key)
- [x] Idempotent Chroma index in `vectorstore/`
- [x] Live query returns page 28 with score ≥ 0.70

**Next:** `notebooks/day2/Day2_Retrieval_Optimization.ipynb`